In [35]:
from pyspark.sql import SparkSession
import os

spark = SparkSession.builder \
    .appName("Local to HDFS Upload") \
    .config("spark.hadoop.fs.defaultFS", "hdfs://hadoop:8020") \
    .getOrCreate()

hdfs = spark.sparkContext._jvm.org.apache.hadoop.fs.FileSystem.get(
    spark.sparkContext._jsc.hadoopConfiguration()
)

local_data_path = os.path.abspath("data")
print(f"Local files: {os.listdir(local_data_path)}")

hdfs_target_path = "/my_data/"

hdfs_path = spark.sparkContext._jvm.org.apache.hadoop.fs.Path(hdfs_target_path)
if not hdfs.exists(hdfs_path):
    hdfs.mkdirs(hdfs_path)
    hdfs.setPermission(
        hdfs_path,
        spark.sparkContext._jvm.org.apache.hadoop.fs.permission.FsPermission("777")
    )
    print(f"Created directory in HDFS: {hdfs_target_path}")

for filename in os.listdir(local_data_path):
    local_file = os.path.join(local_data_path, filename)
    hdfs_file = os.path.join(hdfs_target_path, filename)
    
    print(f"Processing: {local_file} → hdfs://hadoop:8020{hdfs_file}")
    
    if os.path.isfile(local_file):
        try:
            if filename.endswith('.csv'):
                df = spark.read.csv(f"file://{local_file}", header=True, inferSchema=True)
                df.write.mode("overwrite").parquet(f"hdfs://hadoop:8020{hdfs_file}")
            
            elif filename.endswith('.json'):
                df = spark.read.json(f"file://{local_file}")
                df.write.mode("overwrite").json(f"hdfs://hadoop:8020{hdfs_file}")
            
            else:
                src_path = spark.sparkContext._jvm.org.apache.hadoop.fs.Path(f"file://{local_file}")
                dst_path = spark.sparkContext._jvm.org.apache.hadoop.fs.Path(hdfs_file)
                hdfs.copyFromLocalFile(False, True, src_path, dst_path)
            
            print(f"Successfully uploaded {filename} to HDFS")
            
        except Exception as e:
            print(f"Error processing {filename}: {str(e)}")

spark.stop()

25/04/12 21:25:23 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Local files: ['.ipynb_checkpoints', 'columns.json', 'taxi.csv', 'X_test.csv', 'X_train.csv', 'y_test.csv', 'y_train.csv']
Processing: /home/jovyan/dags/data/.ipynb_checkpoints → hdfs://hadoop:8020/my_data/.ipynb_checkpoints
Processing: /home/jovyan/dags/data/columns.json → hdfs://hadoop:8020/my_data/columns.json
Successfully uploaded columns.json to HDFS
Processing: /home/jovyan/dags/data/taxi.csv → hdfs://hadoop:8020/my_data/taxi.csv
Successfully uploaded taxi.csv to HDFS
Processing: /home/jovyan/dags/data/X_test.csv → hdfs://hadoop:8020/my_data/X_test.csv
Successfully uploaded X_test.csv to HDFS
Processing: /home/jovyan/dags/data/X_train.csv → hdfs://hadoop:8020/my_data/X_train.csv
Successfully uploaded X_train.csv to HDFS
Processing: /home/jovyan/dags/data/y_test.csv → hdfs://hadoop:8020/my_data/y_test.csv
Successfully uploaded y_test.csv to HDFS
Processing: /home/jovyan/dags/data/y_train.csv → hdfs://hadoop:8020/my_data/y_train.csv
Successfully uploaded y_train.csv to HDFS
